# Week 3 Day 1 — AFL Data Foundations

EDA, prediction targets, leakage-safe features, and a time-based train/holdout split.

**Outputs**
- `docs/data_dictionary_targets.md`
- `docs/feature_dictionary.md`
- `data/processed/match_features_v1.csv` (+ parquet)
- `data/processed/player_game_features_v1.csv`


## Setup

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path(".").resolve()
sys.path.insert(0, str(ROOT))

from src.load_data import (
    build_match_level,
    inventory_summary,
    load_player_games,
    load_player_seasons,
    load_players_info,
    load_team_matches,
)
from src.targets import TARGET_DICTIONARY, add_match_targets, add_player_targets, IMPACT_WEIGHTS
from src.features import build_match_feature_table, build_player_feature_table
from src.splits import CEILING_NOTE, DEFAULT_HOLDOUT_YEAR, describe_split, time_based_split
from src.paths import FEATURE_MATCH_CSV, FEATURE_PLAYER_CSV, PROCESSED_DIR

sns.set_theme(style="whitegrid", context="notebook")
FIG = ROOT / "docs" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
print("ROOT", ROOT)
print(inventory_summary())


## Task 1 — Data inventory & quality

| Table | Grain | Keys |
|-------|-------|------|
| players_info | player | player_id |
| player round stats | player-game | player_id, team, opponent, match_date |
| player seasonal | player-season | player_id, year, team |
| team matches | team-game (2 rows/match) | team, opponent, match_date, venue |

Derived **match** table: one row per game (home perspective) + `match_id`.


In [ ]:
info = load_players_info()
tm = load_team_matches()
pg = load_player_games()
ps = load_player_seasons()
matches = add_match_targets(build_match_level(tm))

print("years", tm.year.min(), "->", tm.year.max())
print("n teams", tm.team.nunique(), "| venues", tm.venue.nunique())
print("matches", len(matches), "| player-games", len(pg), "| players", info.player_id.nunique())
print("\nMissing % (team matches)")
print((tm.isna().mean()*100).sort_values(ascending=False).head(6).round(2))
print("\nMissing % (player games) top")
print((pg.isna().mean()*100).sort_values(ascending=False).head(10).round(2))
print("\nduplicate player_game_id", pg.player_game_id.duplicated().sum())
print("home_win rate", round(matches.home_win.mean(), 3), "draw rate", round(matches.is_draw.mean(), 3))


In [ ]:
team_years = tm.groupby("team")["year"].agg(["min", "max", "count"]).sort_values("min")
team_years


### Quality flags
- Raw match team names had leading tabs/spaces and `W. Bulldogs` vs `Western Bulldogs` — fixed in `normalize_team`.
- Player `score` is empty in source; use `fantasy_points` / `impact_score`.
- Advanced stats are sparse in older seasons.
- Club changes to watch: Brisbane Bears → Lions, Fitzroy exit, GC & GWS expansion.


## Task 2 — Prediction targets

In [ ]:
pd.DataFrame(TARGET_DICTIONARY)

In [ ]:
print("Impact weights", IMPACT_WEIGHTS)
pg_t = add_player_targets(pg)
print(pg_t[["is_top_disposals", "is_top_goals", "is_top_impact"]].mean().round(4))
matches[["match_result", "home_win", "home_margin"]].head()


**Match winner:** classification on `home_win` (primary). `home_margin` kept for regression. Draws are rare and coded as non-home-win for the binary target.

**Top player:** (1) max disposals in match, (2) max goals in match, (3) composite `impact_score = 3K+2H+3M+4T+6G+1B+1HO`.


## Task 3 — EDA

In [ ]:
season_home = matches.groupby("year").agg(home_win_rate=("home_win", "mean"), n=("match_id", "count"))
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(season_home.index, season_home.home_win_rate, marker="o", ms=3)
ax.axhline(0.5, color="gray", ls="--", lw=1)
ax.set_title("Home win rate by season")
ax.set_ylabel("home_win rate")
ax.set_xlabel("year")
fig.tight_layout()
fig.savefig(FIG / "01_home_win_rate_by_season.png", dpi=120)
plt.show()
season_home.tail()


In [ ]:
log = pd.concat([
    matches.assign(team=matches.home_team, won=matches.home_win),
    matches.assign(team=matches.away_team, won=(1 - matches.home_win) * (1 - matches.is_draw)),
], ignore_index=True)
recent = log[log.year >= 2015]
team_wr = recent.groupby("team").won.mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 5))
team_wr.plot(kind="barh", ax=ax)
ax.set_title("Team win rate (2015 onward)")
ax.set_xlabel("win rate")
fig.tight_layout()
fig.savefig(FIG / "02_team_win_rate_2015.png", dpi=120)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, col in zip(axes, ["disposals", "goals", "impact_score"]):
    s = pg_t[col].dropna()
    ax.hist(s.clip(upper=s.quantile(0.99)), bins=40, color="steelblue", alpha=0.85)
    ax.set_title(col)
fig.suptitle("Player-game distributions")
fig.tight_layout()
fig.savefig(FIG / "03_player_distributions.png", dpi=120)
plt.show()

leaders = (
    pg_t.groupby("player_id")
    .agg(
        disposals=("disposals", "sum"),
        goals=("goals", "sum"),
        impact=("impact_score", "sum"),
        games=("player_game_id", "count"),
    )
    .join(info.set_index("player_id")[["player_name"]], how="left")
)
print("Top disposal career (sum)")
display(leaders.nlargest(8, "disposals")[["player_name", "disposals", "games"]])
print("Top goals career (sum)")
display(leaders.nlargest(8, "goals")[["player_name", "goals", "games"]])


In [ ]:
pg_sorted = pg_t.sort_values(["player_id", "match_date"]).copy()
pg_sorted["disp_prev"] = pg_sorted.groupby("player_id")["disposals"].shift(1)
corr = pg_sorted[["disposals", "disp_prev"]].corr().iloc[0, 1]
print("corr(disposals, previous game) =", round(corr, 3))


In [ ]:
if FEATURE_MATCH_CSV.exists():
    mfeat = pd.read_csv(FEATURE_MATCH_CSV, parse_dates=["match_date"])
else:
    mfeat = build_match_feature_table(build_match_level(tm))

plot_df = mfeat.dropna(subset=["form_margin_diff_l5"]).copy()
plot_df["form_bin"] = pd.qcut(plot_df["form_margin_diff_l5"], 5, duplicates="drop")
form_win = plot_df.groupby("form_bin", observed=True).home_win.mean()

fig, ax = plt.subplots(figsize=(8, 4))
form_win.plot(kind="bar", ax=ax, color="teal")
ax.set_title("Home win rate by form margin diff (L5) quintile")
ax.set_ylabel("home_win rate")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig.savefig(FIG / "04_form_vs_win.png", dpi=120)
plt.show()


In [ ]:
rd = mfeat.dropna(subset=["home_rest_days"]).copy()
rd["rest_bin"] = pd.cut(
    rd["home_rest_days"],
    bins=[-0.1, 5, 7, 9, 14, 60],
    labels=["<=5", "6-7", "8-9", "10-14", "15+"],
)
rest_win = rd.groupby("rest_bin", observed=True).home_win.mean()
fig, ax = plt.subplots(figsize=(7, 4))
rest_win.plot(kind="bar", ax=ax, color="darkorange")
ax.set_title("Home win rate by home rest days")
ax.set_ylabel("home_win rate")
fig.tight_layout()
fig.savefig(FIG / "05_rest_days_vs_win.png", dpi=120)
plt.show()


In [ ]:
inter = mfeat.groupby("is_interstate").home_win.mean()
fig, ax = plt.subplots(figsize=(5, 4))
inter.plot(kind="bar", ax=ax, color=["#4c72b0", "#dd8452"])
ax.set_xticklabels(["same state", "interstate"], rotation=0)
ax.set_title("Home win rate: interstate matchup")
ax.set_ylabel("home_win rate")
fig.tight_layout()
fig.savefig(FIG / "06_interstate_vs_win.png", dpi=120)
plt.show()
print(inter)


In [ ]:
lp = mfeat.dropna(subset=["ladder_pct_diff"]).copy()
lp["ladder_bin"] = pd.qcut(lp["ladder_pct_diff"], 5, duplicates="drop")
ladder_win = lp.groupby("ladder_bin", observed=True).home_win.mean()
fig, ax = plt.subplots(figsize=(8, 4))
ladder_win.plot(kind="bar", ax=ax, color="slateblue")
ax.set_title("Home win rate by ladder_pct_diff quintile (pre-match)")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig.savefig(FIG / "07_ladder_diff_vs_win.png", dpi=120)
plt.show()


In [ ]:
h2 = mfeat[mfeat.h2h_games >= 3].copy()
h2["h2h_bin"] = pd.qcut(h2["h2h_home_win_rate"], 4, duplicates="drop")
h2w = h2.groupby("h2h_bin", observed=True).home_win.mean()
fig, ax = plt.subplots(figsize=(7, 4))
h2w.plot(kind="bar", ax=ax, color="seagreen")
ax.set_title("Home win rate by prior H2H win-rate quartile")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig.savefig(FIG / "08_h2h_vs_win.png", dpi=120)
plt.show()


## Task 4 — Feature engineering (no leakage)

Rolling / ladder / H2H features use **shift(1)** so the current match is never included.
Rebuild with: `python build_features.py`


In [ ]:
if FEATURE_MATCH_CSV.exists():
    match_features = pd.read_csv(FEATURE_MATCH_CSV, parse_dates=["match_date"])
else:
    match_features = build_match_feature_table(build_match_level(tm))
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    match_features.to_csv(FEATURE_MATCH_CSV, index=False)

print(match_features.shape)
match_features.head(3)


In [ ]:
if FEATURE_PLAYER_CSV.exists():
    player_features = pd.read_csv(FEATURE_PLAYER_CSV, parse_dates=["match_date"], nrows=5000)
    print("preview rows", len(player_features), "(full file on disk)")
else:
    player_features = build_player_feature_table(pg)
print(player_features.columns.tolist())
player_features.head(3)


Feature dictionary: `docs/feature_dictionary.md`.

## Task 5 — Time-based split

In [ ]:
split_info = describe_split(match_features, holdout_year=DEFAULT_HOLDOUT_YEAR)
print(split_info)
train, holdout = time_based_split(match_features, holdout_year=DEFAULT_HOLDOUT_YEAR)
print("train home_win rate", round(train.home_win.mean(), 3))
print("holdout home_win rate", round(holdout.home_win.mean(), 3))
print()
print(CEILING_NOTE)


### Why not a random split?
Sports rows are time-ordered. A random split can put a team's future games in train and past games in test, so evaluation leaks information you would not have on a live tip.

### Accuracy ceiling
See printed note above — roughly 60–70% winner accuracy is a realistic long-run band; near-perfect holdout scores usually mean leakage.


## Done
Day 2 should import `time_based_split` and the v1 feature tables without redefining targets.
